In [449]:
import pandas as pd

In [450]:
# 외부의 텍스트파일을 로드하는 방법
text = open('../data_git/data_NLP/qa.txt', 'r', encoding='utf-8').read()

In [451]:
# 일반 문자를 python의 구조로 변경하는 함수
QnA_list = eval(text)
type(QnA_list)

list

In [452]:
QnA_list

[('환불은 어떻게 하나요?', "주문 상세 페이지에서 '환불 신청' 버튼을 눌러 접수하실 수 있습니다."),
 ('배송 기간은 얼마나 걸리나요?', '일반 배송은 2~3일, 도서산간 지역은 최대 5일까지 소요됩니다.'),
 ('해외 배송도 가능한가요?', '현재 해외 배송은 지원하지 않습니다.'),
 ('회원 탈퇴는 어디에서 하나요?', '설정 > 계정 관리 > 회원 탈퇴 메뉴에서 진행하실 수 있습니다.'),
 ('비밀번호를 잊어버렸어요', "로그인 화면의 '비밀번호 재설정' 링크를 통해 재설정 가능합니다."),
 ('주문 취소는 어떻게 하죠?', '상품이 배송 준비 전 상태라면 주문 상세 페이지에서 취소가 가능합니다.'),
 ('영수증 발급이 가능한가요?', '마이페이지 > 주문 내역에서 영수증 출력이 가능합니다.'),
 ('교환/반품은 가능한가요?', '수령일로부터 7일 이내, 미사용/미훼손 제품에 한해 가능합니다.')]

- QnA_list 데이터에서 질문들을 따로 추출하고 형태소 분석을 통해 단어를 추출
- 벡터화 작업 (TF-IDF)
- 코사인 유사도
    - 문장과 문장 사이에 어느정도 같은 의미를 가지는가?

In [453]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from konlpy.tag import Komoran

In [454]:
# QnA_list에서 질문들의 목록을 생성
# 패턴 인지 -> list에서 각각의 원소들을 추출 -> 원소에서 첫 번째 문자를 추출
# 목록이라는 새로운 리스트를 생성
# 방법1
q=[]
for s in QnA_list:
    q.append(s[0])
q

['환불은 어떻게 하나요?',
 '배송 기간은 얼마나 걸리나요?',
 '해외 배송도 가능한가요?',
 '회원 탈퇴는 어디에서 하나요?',
 '비밀번호를 잊어버렸어요',
 '주문 취소는 어떻게 하죠?',
 '영수증 발급이 가능한가요?',
 '교환/반품은 가능한가요?']

In [455]:
# 방법2
questions = [ q for q, a in QnA_list ]
questions

['환불은 어떻게 하나요?',
 '배송 기간은 얼마나 걸리나요?',
 '해외 배송도 가능한가요?',
 '회원 탈퇴는 어디에서 하나요?',
 '비밀번호를 잊어버렸어요',
 '주문 취소는 어떻게 하죠?',
 '영수증 발급이 가능한가요?',
 '교환/반품은 가능한가요?']

In [456]:
# 토큰화 -> 벡터화
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 2),
    lowercase=False
)

In [457]:
X = vectorizer.fit_transform(questions)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [458]:
X.toarray().shape

(8, 76)

In [459]:
# 새로운 질문
query = '환불을 하려면 어떻게 하면 될까요?'
# 질문을 벡터화
query_vec = vectorizer.transform([query])

In [460]:
query_vec.shape

(1, 76)

In [461]:
# 코사인 거리 유사도 함수를 사용
# ravel() -> array에서 사용하는 함수로 다차원 배열을 1차원 배열로 변경하는 함수
sims = cosine_similarity(query_vec, X).ravel()

In [462]:
sims.shape

(8,)

In [463]:
# argsort(): 배열의 값을 정렬했을때 그 정렬 순서를 되돌려주는 함수
# 정렬의 순서는 인덱스를 의미
rank = sims.argsort()[::-1]

In [464]:
print('질문 : ', query)
for i in rank[:2]:
    print(f"index : {i}, 유사도 : {round(sims[i], 3)}, 유사질문 : {questions[i]}, 답변 : {QnA_list[i][1]}")

질문 :  환불을 하려면 어떻게 하면 될까요?
index : 0, 유사도 : 0.69, 유사질문 : 환불은 어떻게 하나요?, 답변 : 주문 상세 페이지에서 '환불 신청' 버튼을 눌러 접수하실 수 있습니다.
index : 5, 유사도 : 0.453, 유사질문 : 주문 취소는 어떻게 하죠?, 답변 : 상품이 배송 준비 전 상태라면 주문 상세 페이지에서 취소가 가능합니다.


In [465]:
df1 = pd.read_json('../data_git/data_NLP/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json')
df2 = pd.read_json('../data_git/data_NLP/민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json')

In [466]:
df1.head(5)

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"


### 문제
1. 일반행정 데이터, 대중교통 데이터 로드
2. 두 개의 데이터프레임을 결합 ()
3. 데이터의 필터링. 고객질문에 대한 상담사의 답변이 즉각적으로 오는 데이터들만 필터
4. 질문 중 중복 데이터를 제거
5. 질문들을 모아서 토큰화, 벡터화
6. 그 외의 질문 목록을 이용하여 코사인 유사도를 확인하고 유사 질문과 답변을 출력

## 문제1 - 방법1

In [467]:
df = pd.concat([df1, df2], ignore_index=True)

In [468]:
# 고객질문(요청) zjffjadml epdlxjemfdml rotnfmf ghkrdls
df['고객질문(요청)'].value_counts()

고객질문(요청)
                             58725
카드결제와 현금결제할 때 요금 차이가 있나요?       79
                                78
버스요금은 얼마인가요?                    55
시간은 얼마나 걸려요?                    54
                             ...  
국도로 빠지면 가까운 주유소가 있나요?            1
정체가 해소되길 기다리는게 낫겠죠?              1
km는 별 차이 없나요?                    1
지금 그냥 국도로 빠져서 가는게 더 빠를까요?        1
그럼 순천 시내버스 막차는 몇 시 인가요?          1
Name: count, Length: 22824, dtype: int64

In [469]:
# 공백의 데이터가 여러개 존재하므로 공백으로 이루어진 value들을 통일화
df['고객질문(요청)'] = df['고객질문(요청)'].str.strip()

In [470]:
df = df[['고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변']]
df.head(10)

,고객질문(요청),상담사질문(요청),고객답변,상담사답변
0,지방세를 내려면 어떻게 해야됩니까?,,,
1,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.
2,은행 어플에서도 됩니까?,,,
3,,어떤 은행을 이용하고 계십니까?,,
4,,,기업은행을 이용하고 있습니다.,
5,,,,그럼 스마트폰에서 기업은행 어플을 설치하시면 납부가 가능합니다.
6,은행을 직접방문해도 됩니까?,,,
7,,,,방문납부도 가능합니다.
8,은행위치 좀 알 수 있습니까?,,,
9,,어느지점으로 안내해드릴까요?,,


In [471]:
df.iloc[3, 3] == ''

True

In [472]:
q_list = []
for i in range(len(df)):
    if df.iloc[i, 0] != '':
        q_list.append(i)
        
len(q_list)

30495

In [473]:
a_index = []
q_index = []
for q in q_list:
    if df.iloc[(q+1), 3] != '':
        a_index.append(q+1)
        q_index.append(q)

print(len(a_index))
print(len(q_index))

25455
25455


In [474]:
question = df.iloc[q_index, :]['고객질문(요청)'].values
answer = df.iloc[a_index, :]['상담사답변'].values

df = pd.DataFrame({'question' : question, 
                   'answer' : answer})
df.drop_duplicates(subset = 'question', inplace=True)
df.reset_index(drop=True, inplace=True)

In [475]:
df['question'] = df['question'].str.strip()

In [476]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18957 entries, 0 to 18956
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  18957 non-null  object
 1   answer    18957 non-null  object
dtypes: object(2)
memory usage: 296.3+ KB


In [477]:
# 토큰화 -> 벡터화
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)


In [478]:
vector = TfidfVectorizer(tokenizer=tokenize)
question = df['question'].values
X = vector.fit_transform(question)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [479]:
new_questions = [
    '여권 재발급 신청 방법을 알려줘',
    '전입 신고가 인터넷으로 가능한가요?',
    '지방세 환급금을 어디서 신청하나요?'
]

In [480]:
new_questions_vec = vector.transform(new_questions)

In [481]:
sims = cosine_similarity(new_questions_vec[2], X).ravel()
rank = sims.argsort()[::-1]

In [482]:
print('질문 : ',  new_questions[2])
for i in rank[:2]:
    print(f"index : {i}, 유사도 : {round(sims[i], 3)}, 유사질문 : {df['question'][i]}, 답변 : {df['answer'][i]}")

질문 :  지방세 환급금을 어디서 신청하나요?
index : 8786, 유사도 : 0.761, 유사질문 : 지방세 환급금 신청은 어떻게 해야하죠?, 답변 : 인터넷에서 접수를 하셔야 합니다
index : 8792, 유사도 : 0.568, 유사질문 : 환급금을 기부할 수도 있나요?, 답변 : 네 환급금을 사회복지공동모금회에 본인 명의로 기부가 가능합니다


## 문제1 - 방법2

In [516]:
# 2개의 데이터 프레임을 단순한 행결합 (union 결합)
total_df = pd.concat([df1, df2], ignore_index=True)

In [517]:
# 시리즈에서 각각의 value를 추출하여 함수에 대입 -> map()
total_df = total_df.map(
    lambda x : str(x).strip()
)

In [518]:
# 1번 조건식 -> 현재 행에서 고객질문(요청) 데이터가 ''가 아니고
#          -> 다음 행의 상담사답변의 value가 ''이 아닌 경우
flag1 = (total_df['고객질문(요청)'] != '') & (total_df['상담사답변'].shift(-1) != '')

In [519]:
# 2번 조건식 -> 현재 행에서 상담사 답변이 ''가 아니고
#          -> 전 행의 고객질문(요청) 데이터가 ''가 아닌 경우
flag2 = (total_df['상담사답변'] != '') & (total_df['고객질문(요청)'].shift(1) != '')

In [520]:
total_df = total_df.loc[flag1 | flag2, ]

In [521]:
total_df['상담사답변'] = total_df['상담사답변'].shift(-1)

In [522]:
# 고객질문(요청) 데이터에서 ''가 아닌 데이터만 필터
total_df = total_df.loc[
    total_df['고객질문(요청)'] != ''
]

In [523]:
# 고객 질문 데이터중 중복 데이터는 제거
total_df = total_df.drop_duplicates('고객질문(요청)').reset_index(drop=True)

In [491]:
total_df.to_csv('민원 질의응답(즉답형데이터).csv', index=False)

In [492]:
# 토큰화, 벡터화 정의
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize, 
    lowercase=False, 
    ngram_range= (1,1), 
    min_df = 5, 
    max_df= 0.8
)

In [493]:
# total_df에서 고객질문(요청) 데이터를 토큰화, 백터화 작업 
X = vectorizer.fit_transform(
    total_df['고객질문(요청)'].values
)

In [494]:
# new_questions도 토큰, 백터화 -> fit() x
test = vectorizer.transform(new_questions)

In [495]:
# 코사인 유사도 생성 
sims = cosine_similarity(test, X)

In [496]:
for idx, sim in enumerate(sims):
    question = new_questions[idx]
    print("유저의 질문 : ", question)
    # sim데이터에서 내림차순정렬을 한 인덱스의 목록 
    sim_idxs = sim.argsort()[::-1]
    for i in sim_idxs[:2]:
        # i : 유저의 질문에 가장 유사한 질문의 인덱스
        print(f"유사도 : {round(sim[i], 3)} \
              유사 질문 : {total_df.loc[i, '고객질문(요청)']}, \
              답변 : {total_df.loc[i, '상담사답변']}")

유저의 질문 :  여권 재발급 신청 방법을 알려줘
유사도 : 0.619               유사 질문 : 신청방법을 알려주세요.,               답변 : 주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
유사도 : 0.568               유사 질문 : 신청방법 좀 알려주세요?,               답변 : 우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.
유저의 질문 :  전입 신고가 인터넷으로 가능한가요?
유사도 : 0.738               유사 질문 : 인터넷으로도 신고가능한가요?,               답변 : 방문 접수밖에 안됩니다.
유사도 : 0.682               유사 질문 : 인터넷으로 가능한가요?,               답변 : 인터넷으로 신청 가능합니다.
유저의 질문 :  지방세 환급금을 어디서 신청하나요?
유사도 : 0.76               유사 질문 : 지방세 환급금 신청은 어떻게 해야하죠?,               답변 : 인터넷에서 접수를 하셔야 합니다
유사도 : 0.566               유사 질문 : 환급금을 기부할 수도 있나요?,               답변 : 네 환급금을 사회복지공동모금회에 본인 명의로 기부가 가능합니다


### 문제 2
- 고객질문의 데이터를 이용하여 카테고리를 분류하는 모델을 생성
    - 고객질문 데이터들을 이용하여 토큰화, 벡터화 작업 (독립 변수)
    - 카테고리 일반행정, 대중교통을 타겟 데이터 (종속 변수)
        - 카테고리 데이터를 LabelEncoder()을 이용하여 수치화 변환
    - svc 모델을 이용하여 벡터화된 데이터와 카테고리 데이터를 이용하여 학습
    - new_question의 카테고리를 확인

In [497]:
# 데이터프레임 결합
df_cg = pd.concat((df1[['카테고리', '고객질문(요청)']], df2[['카테고리', '고객질문(요청)']]))

In [498]:
# 이상치 제거
df_cg['고객질문(요청)'] = df_cg['고객질문(요청)'].str.strip()
df_cg = df_cg.loc[df_cg['고객질문(요청)'] != '']

In [499]:
df_cg

,카테고리,고객질문(요청)
0,일반행정 문의,지방세를 내려면 어떻게 해야됩니까?
2,일반행정 문의,은행 어플에서도 됩니까?
6,일반행정 문의,은행을 직접방문해도 됩니까?
8,일반행정 문의,은행위치 좀 알 수 있습니까?
12,일반행정 문의,버스로 가는 방법도 있습니까?
...,...,...
38934,대중교통 안내,네 알겠습니다. 뭐 하나 더 물어봐도 돼요?
38936,대중교통 안내,그럼 익산 시내버스 막차는 몇 시 인가요?
38946,대중교통 안내,여기가 대중교통 관련 질문 센터 맞나요?
38950,대중교통 안내,순천 시내버스 첫차가 몇 시 인가요?


In [500]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_cg['카테고리'] = le.fit_transform(df_cg['카테고리'])

In [501]:
df_cg

,카테고리,고객질문(요청)
0,1,지방세를 내려면 어떻게 해야됩니까?
2,1,은행 어플에서도 됩니까?
6,1,은행을 직접방문해도 됩니까?
8,1,은행위치 좀 알 수 있습니까?
12,1,버스로 가는 방법도 있습니까?
...,...,...
38934,0,네 알겠습니다. 뭐 하나 더 물어봐도 돼요?
38936,0,그럼 익산 시내버스 막차는 몇 시 인가요?
38946,0,여기가 대중교통 관련 질문 센터 맞나요?
38950,0,순천 시내버스 첫차가 몇 시 인가요?


In [502]:
X = df_cg['고객질문(요청)'].values
Y = df_cg['카테고리'].values

In [503]:
# 토큰화 -> 벡터화
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 2),
    lowercase=False
)

X = vectorizer.fit_transform(X)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
from sklearn.svm import SVC
svc = SVC()

In [ ]:
svc.fit(X, Y)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [ ]:
X_test = vectorizer.transform(new_questions)

In [ ]:
pred = svc.predict(X_test)

print(pred)

[1 1 1]


In [ ]:
le.inverse_transform(pred)

### 문제2 - 방법2
- train, test 분할

In [424]:
X = total_df['고객질문(요청)']
Y = total_df['카테고리']

In [425]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

In [426]:
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer2 = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 2),
    lowercase=False
)

X_train = vectorizer2.fit_transform(X_train)
X_test = vectorizer2.transform(X_test)

/Users/eunseo/Documents/data_boot/venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [427]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
Y_train_le = le.fit_transform(Y_train)
Y_test_le = le.transform(Y_test)

In [429]:
svc2 = SVC()
svc2.fit(X_train, Y_train_le)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [430]:
pred = svc2.predict(X_test)

In [431]:
from sklearn.metrics import accuracy_score
acc = accuracy_score(pred, Y_test_le)

In [432]:
acc

0.8881561593247165

In [504]:
# 질문 목록
new_questions = [
    '여권 재발급 신청 방법을 알려줘',
    '전입 신고가 인터넷으로 가능한가요?',
    '지방세 환급금을 어디서 신청하나요?',
    '서울역에서 영등초로 가려면 어떻게 가나요?'
]

- new_questions 데이터를 이용하여 svc 모델로 예측
- 예측 값을 이용하여 total_df 의 카테고리 필터링
- 고객질문 모음을 토큰화, 벡터화 작업
- new_question의 유사한 질문과 답변을 상위 2개만 출력

In [505]:
X_test = vectorizer2.transform(new_questions)

In [506]:
pred_new = svc2.predict(X_test)

In [512]:
pred_new_origin = le.inverse_transform(pred_new)
pred_new_origin

array(['일반행정 문의', '일반행정 문의', '일반행정 문의', '대중교통 안내'], dtype=object)

In [508]:
# new_question 벡터화, 토큰화
test = vectorizer.transform(new_questions)

In [509]:
sims = cosine_similarity(test, X)

In [514]:
sims.shape

(4, 30495)

In [537]:
i = 0
for vec_data, cate in zip(test, pred_new_origin):
    # cate를 기준으로 total_df 에서 카테고리 필터링 -> 벡터화 작업
    x_train = vectorizer.transform(
        total_df.loc[total_df['카테고리'] == cate, '고객질문(요청)']
    )
    # 코사인 유사도 (질문이 하나) -> 2차원 데이터를 1차원으로 변경 필요
    sims = cosine_similarity(vec_data, x_train).ravel()
    # 유사도를 내림차순 정렬의 형태로 인덱스의 값들을 확인
    idxs = sims.argsort()[::-1]
    # 유사도 리스트에서 유사도가 높은 상위 2개만 출력하여 유사 질문 답변을 출력
    print('\n', '질문 : ', new_questions[i])
    i = i+1

    for idx in idxs[:2]:
        # idx는 array의 위치값
        # 카테고리별로 필터링된 데이터프레임에서 array와 같이 인덱스는 위치로 변환

        # iloc를 이용하여 유사 질문 출력
        print("유사질문 : ", total_df.loc[total_df['카테고리'] == cate, '고객질문(요청)'].iloc[idx])
        # 답변 출력
        print("답변 : ", total_df.loc[total_df['카테고리'] == cate, '상담사답변'].iloc[idx])


 질문 :  여권 재발급 신청 방법을 알려줘
유사질문 :  신청방법을 알려주세요.
답변 :  주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
유사질문 :  신청방법 좀 알려주세요?
답변 :  우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.

 질문 :  전입 신고가 인터넷으로 가능한가요?
유사질문 :  인터넷으로 가능한가요?
답변 :  인터넷으로 신청 가능합니다.
유사질문 :  납부는 인터넷으로 가능한가요?
답변 :  네. 가능합니다.

 질문 :  지방세 환급금을 어디서 신청하나요?
유사질문 :  지방세 환급금 신청은 어떻게 해야하죠?
답변 :  인터넷에서 접수를 하셔야 합니다
유사질문 :  어디서 신청하나요?
답변 :  온라인청년센터 통해서 신청가능합니다!

 질문 :  서울역에서 영등초로 가려면 어떻게 가나요?
유사질문 :  어떻게 가나요?
답변 :  오류역에서 지하철을 탄 후 대전역 지하철에서 내리신 후 14번 버스를 타면됩니다.
유사질문 :  그럼 어떻게 가나요?
답변 :  여의도역 인근까지 가셔서 도보로 이동하셔야 합니다.


In [ ]:
for idx, sim in enumerate(sims):
    question = new_questions[idx]
    print("유저의 질문 : ", question)
    # sim데이터에서 내림차순정렬을 한 인덱스의 목록 
    sim_idxs = sim.argsort()[::-1]
    for i in sim_idxs[:2]:
        # i : 유저의 질문에 가장 유사한 질문의 인덱스
        print(f"유사도 : {round(sim[i], 3)} \
              유사 질문 : {total_df.loc[i, '고객질문(요청)']}, \
              답변 : {total_df.loc[i, '상담사답변']}")